In [ ]:
# ── CELL 1: INSTALLATION ──────────────────────────────────
# Estimated time: 2-3 min. Wait for "DONE".

# 1. Install system dependencies & Ollama
!apt-get install -y zstd curl lsof -q 2>/dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Install Open WebUI
!pip install open-webui -q

# 3. Install Cloudflare Tunnel (cloudflared)
import os, shutil
CF_BIN = "/usr/local/bin/cloudflared"
if not shutil.which("cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o {CF_BIN} && chmod +x {CF_BIN}
else:
    print("  ✅ cloudflared already installed")

# 4. Verify all dependencies
missing = []
for cmd in ["ollama", "cloudflared", "open-webui"]:
    if not shutil.which(cmd):
        missing.append(cmd)
if missing:
    print(f"❌ FAILED: {', '.join(missing)} not found after installation")
else:
    print("────────────────────────────────")
    print("✅ DONE — All dependencies installed. Proceed to Cell 2.")
    print("────────────────────────────────")

In [ ]:
# ── CELL 2: SETUP & RUN OLLAMA ────────────────────────────
# Change model below if needed:
MODEL = "qwen2.5-coder:7b"   # default (~4.7GB)
# MODEL = "qwen2.5-coder:14b"  # coding (~9.0GB)
# MODEL = "huihui_ai/qwen2.5-coder-abliterate:14b-instruct-q4_k_m"  # uncensored coding (~9.0GB)
# MODEL = "supergoatscriptguy/mythos-sec:24b"  # pentest/CTF uncensored (~14GB)
# MODEL = "dolphin-llama3:8b"  # uncensored general/agentic (~4.7GB)
# MODEL = "deepseek-coder:14b" # coding (~8.5GB)
# MODEL = "deepseek-r1:7b"     # reasoning (~4.7GB)
# MODEL = "gemma4:12b"         # reasoning & coding (~6.6GB)
# MODEL = "llama3.1:8b"        # general chat (~4.9GB)

import subprocess, time, os, shutil, json, urllib.request, sys
from google.colab import drive

# ─── Check prerequisites first ───
ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
if not os.path.exists(ollama_bin):
    print("❌ ERROR: Ollama not found. Run Cell 1 first.", flush=True)
    sys.exit(1)

# ─── Mount Google Drive (only if not already) ───
print("[1/4] Mounting Google Drive...", flush=True)
if not os.path.exists("/content/drive"):
    drive.mount('/content/drive')
else:
    print("  ✅ Drive already mounted")

# ─── Set up model paths ───
print("[2/4] Setting up model folder...")
DRIVE_PATH = "/content/drive/MyDrive/ollama_models"
LOCAL_PATH = os.path.expanduser("~/.ollama/models")

# Restore model from Drive to local (if exists)
RESTORED_FLAG = os.path.join(LOCAL_PATH, ".restored")
if os.path.exists(DRIVE_PATH) and os.listdir(DRIVE_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)
    if not os.path.exists(RESTORED_FLAG):
        print("  📦 Restoring model from Google Drive...")
        for item in os.listdir(DRIVE_PATH):
            src = os.path.join(DRIVE_PATH, item)
            dst = os.path.join(LOCAL_PATH, item)
            if not os.path.exists(dst):
                try:
                    if os.path.isdir(src):
                        shutil.copytree(src, dst, dirs_exist_ok=True)
                    else:
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"    ⚠️  Failed to copy {item}: {e}")
        open(RESTORED_FLAG, 'w').close()
        print("  ✅ Model restored from Drive")
    else:
        print("  ✅ Model already restored (skip)")
elif not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH, exist_ok=True)

# Use local path (not Drive directly) — more compatible with Ollama
os.environ["OLLAMA_MODELS"] = LOCAL_PATH

# ─── Kill stale Ollama processes (if any) ───
print("[3/4] Cleaning up stale processes...")
os.system("pkill -f 'ollama serve' 2>/dev/null || true")
time.sleep(1)

# ─── Start Ollama ───
print("[4/4] Starting Ollama & loading model...")

# Bind to 127.0.0.1 — more secure, tunnel still works
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
proc_ollama = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait for Ollama to be ready
print("  Waiting for Ollama to be ready...", end="", flush=True)
ollama_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        r.close()
        print(" ✅")
        ollama_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
if not ollama_ready:
    print(" ⚠️  Ollama not responding — check logs above")

# Pull model
print(f"  Downloading model {MODEL} (may take several minutes)...")
pull = subprocess.Popen(
    [ollama_bin, "pull", MODEL],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(pull.stdout.readline, ''):
    line = line.rstrip()
    if line:
        print(f"  {line}")
pull.wait()

if pull.returncode != 0:
    print(f"  ⚠️  Pull failed (exit code {pull.returncode})")
else:
    print("  ✅ Model pulled successfully")

# Backup model to Drive
if os.path.exists(LOCAL_PATH):
    print("  💾 Backing up model to Google Drive...")
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(LOCAL_PATH):
        if item.startswith('.'):
            continue
        src = os.path.join(LOCAL_PATH, item)
        dst = os.path.join(DRIVE_PATH, item)
        if not os.path.exists(dst):
            try:
                if os.path.isdir(src):
                    shutil.copytree(src, dst, dirs_exist_ok=True)
                else:
                    shutil.copy2(src, dst)
            except Exception as e:
                print(f"    ⚠️  Failed to backup {item}: {e}")
    print("  ✅ Backup to Drive complete")

print("────────────────────────────────")
print(f"✅ MODEL {MODEL} READY — Proceed to Cell 3.")
print("────────────────────────────────")

In [ ]:
# ── CELL 3: RUN WEBUI & TUNNEL (CLOUDFLARE) ────────────────
# THIS CELL MUST KEEP RUNNING WHILE USING WEBUI
import os, subprocess, time, urllib.request, re, signal, shutil, sys, threading

PORT = 8081

# ─── Helper: consume cloudflared stdout in a thread ───
def consume_stdout(proc, on_line=None):
    """Read proc stdout line by line, call callback, print to console."""
    for line in iter(proc.stdout.readline, ''):
        print(line, end="", flush=True)
        if on_line:
            on_line(line)

# ─── Helper: kill process on a given port ───
def kill_port(port):
    methods = [
        f"fuser -k {port}/tcp 2>/dev/null",
        f"lsof -ti:{port} | xargs kill -9 2>/dev/null",
        f"ss -tlnp | grep ':{port}' | tr -s ' ' | cut -d' ' -f7 | cut -d',' -f2 | xargs kill -9 2>/dev/null"
    ]
    for cmd in methods:
        ret = os.system(cmd)
        if os.WIFEXITED(ret) and os.WEXITSTATUS(ret) == 0:
            return True
    return False

# ─── 1. Find open-webui binary ───
webui_bin = shutil.which("open-webui") or shutil.which("open_webui")
if not webui_bin:
    print("❌ open-webui not found. Run Cell 1 first.", flush=True)
    raise SystemExit(1)

# ─── 2. Kill port if stale ───
print(f"[1/4] Cleaning up port {PORT}...", flush=True)
kill_port(PORT)
time.sleep(1)

# ─── 3. Start Open WebUI ───
print(f"[2/4] Starting Open WebUI on port {PORT}...", flush=True)
env = {**os.environ, "OLLAMA_BASE_URL": "http://127.0.0.1:11434"}
proc_webui = subprocess.Popen(
    [webui_bin, "serve", "--port", str(PORT)],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# ─── 4. Wait for WebUI to be ready ───
print("[3/4] Waiting for WebUI to be ready (initial load 1-3 min)...", flush=True)
webui_ready = False
for i in range(60):
    if proc_webui.poll() is not None:
        print(f"\n❌ WebUI failed to start (exit code {proc_webui.returncode})", flush=True)
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://127.0.0.1:{PORT}", timeout=5)
        r.close()
        print("\n✅ WebUI is running in background!", flush=True)
        webui_ready = True
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

if not webui_ready:
    print("\n⚠️  Timeout — tunnel will still be attempted.", flush=True)

# ─── 5. Cloudflare Tunnel (WebUI) ───
print("[4/4] Creating public link via Cloudflare Tunnel...\n", flush=True)

webui_url = None

def on_webui_line(line):
    global webui_url
    m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if m and webui_url is None:
        webui_url = m.group()

proc_tunnel_webui = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

t_webui = threading.Thread(
    target=consume_stdout,
    args=(proc_tunnel_webui, on_webui_line),
    daemon=True
)
t_webui.start()

# Wait for tunnel URL (max 30 seconds)
print("⏳ Waiting for Cloudflare tunnel", end="", flush=True)
for i in range(30):
    if webui_url:
        print()
        break
    if i % 5 == 0:
        print(".", end="", flush=True)
    time.sleep(1)
else:
    print(" timeout.", flush=True)

if webui_url:
    print(flush=True)
    print("═" * 60, flush=True)
    print(" 🌐 OPEN THE LINK BELOW IN YOUR BROWSER:", flush=True)
    print(f" 👉 {webui_url}", flush=True)
    print("═" * 60, flush=True)
    print("Cloudflare Tunnel works immediately, no need to click 'Continue'.", flush=True)

# ─── Keep-Alive: monitor both processes ───
print("\n📡 Monitoring WebUI & Tunnel (Ctrl+C to stop)...", flush=True)
try:
    while True:
        time.sleep(30)
        if proc_webui.poll() is not None:
            print("⚠️  WebUI process has stopped! Re-run Cell 3.", flush=True)
            break
        if proc_tunnel_webui.poll() is not None:
            print("⚠️  Cloudflare Tunnel disconnected! Re-run Cell 3.", flush=True)
            break
        print("✓ WebUI & Tunnel active —", time.strftime("%H:%M:%S"), flush=True)
except KeyboardInterrupt:
    print("\nProcess stopped manually.", flush=True)

In [ ]:
# ── CELL 4: OLLAMA API TUNNEL + OPENCODE CONFIG ───────────
# Run this cell AFTER Cell 3 (WebUI) is running.
# This cell creates a tunnel for the Ollama API (port 11434) for opencode CLI.
import subprocess, re, os, shutil, time, threading, sys

OLLAMA_PORT = 11434

# ─── Helper ───
def consume_stdout(proc, on_line=None):
    for line in iter(proc.stdout.readline, ''):
        print(line, end="", flush=True)
        if on_line:
            on_line(line)

# ─── Verify Ollama is running ───
import urllib.request
print("⏳ Checking Ollama API connection...", flush=True)
try:
    r = urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3)
    r.close()
    print("✅ Ollama API active on port 11434", flush=True)
except Exception:
    print("⚠️  Ollama not responding — make sure Cell 2 has been run.", flush=True)
    # Continue anyway, might connect later

# ─── 1. Tunnel for Ollama API ───
print("\n[1/2] Creating tunnel for Ollama API...", flush=True)

ollama_url = None

def on_ollama_line(line):
    global ollama_url
    m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if m and ollama_url is None:
        ollama_url = m.group()

proc_tunnel_ollama = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{OLLAMA_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

t_ollama = threading.Thread(
    target=consume_stdout,
    args=(proc_tunnel_ollama, on_ollama_line),
    daemon=True
)
t_ollama.start()

# Wait for tunnel URL (max 30 seconds)
print("⏳ Waiting for Cloudflare tunnel", end="", flush=True)
for i in range(30):
    if ollama_url:
        print()
        break
    if i % 5 == 0:
        print(".", end="", flush=True)
    time.sleep(1)
else:
    print(" timeout.")

if not ollama_url:
    print("\n❌ Failed to get tunnel URL for Ollama API.", flush=True)
    print("   Check: is cloudflared installed? Internet connection?", flush=True)
    sys.exit(1)
else:
    print("═" * 60, flush=True)
    print("🔗 OLLAMA API PUBLIC URL (for opencode):", flush=True)
    print(f"   {ollama_url}", flush=True)
    print("═" * 60, flush=True)
    print(flush=True)

    # ─── 2. Display opencode configuration ───
    print("[2/2] opencode.json configuration", flush=True)
    print("═" * 60, flush=True)
    print("Create opencode.json in your project folder:\n", flush=True)

    # Version with /v1 (REQUIRED for OpenAI-compatible API)
    print("{", flush=True)
    print(f'  "endpoint": "{ollama_url}/v1",', flush=True)
    print('  "model": "' + MODEL + '",', flush=True)
    print('  "api_key": "",', flush=True)
    print('  "provider": "openai"', flush=True)
    print("}", flush=True)
    print(flush=True)
    print("📝 Notes:", flush=True)
    print("   - Ollama API tunnel MUST be running for opencode to connect.", flush=True)
    print("   - If Colab disconnects, re-run Cell 2 → Cell 3 → Cell 4.", flush=True)
    print("   - Change model to match the one selected in Cell 2.", flush=True)
    print("   - ⚠️  This URL is PUBLIC — do not share it!", flush=True)

# ─── Keep-Alive ───
print("\n📡 Ollama API tunnel active (Ctrl+C to stop)...", flush=True)
try:
    while True:
        time.sleep(60)
        if proc_tunnel_ollama.poll() is not None:
            print("⚠️  Ollama tunnel disconnected! Re-run Cell 4.", flush=True)
            break
        print("✓ Tunnel —", time.strftime("%H:%M:%S"), flush=True)
except KeyboardInterrupt:
    print("\nTunnel stopped.", flush=True)